# E8 y E13: LSTM + MLP

Cada punto de decisión recibe las sesiones de [*t* − 24 h, *t*) como secuencia (hasta 256, con máscara), recalculada desde cero en cada punto; el último paso es la sesión abierta, sin sus apps. El último estado de la LSTM se une al perfil, la hora y el contexto en un MLP.

Criterio fijado antes (`README.md`, fase F): la LSTM se adopta solo si supera al mejor modelo tabular por 0.01 de AUC o más (mediana por persona, dejando una persona fuera). E13: MPUD se queda solo si suma 0.01 o más.

**Duración en esta máquina (CPU):** unos 6 min por época de MPUD (hasta 30, con parada temprana) y 18 pliegues × 2 brazos de ajuste fino. En total, entre 4 y 8 horas.

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.modeling import load, feature_cols, persistence_score, BLOCKS
from experiments import tabular as T
import torch
from experiments import lstm as L
torch.set_num_threads(os.cpu_count())
pd.set_option("display.width", 160)

## Datos

In [2]:
real, mpud = load("real"), load("mpud")
cols = feature_cols(real)
print(f"real: {len(real)} examples, {real.user_id.nunique()} people, y = {real.y.mean():.1%}")
print(f"mpud: {len(mpud)} examples, {mpud.user_id.nunique()} people, y = {mpud.y.mean():.1%}")
print(f"{len(cols)} features")
dates = ["start_utc", "start_local", "end_utc"]
sessions = pd.concat([pd.read_csv(f"data/processed/{d}_preprocess_3.csv", parse_dates=dates)
                      for d in ("mpud", "real")], ignore_index=True)
log = L.SessionLog(sessions)

real: 16281 examples, 18 people, y = 43.9%
mpud: 193980 examples, 196 people, y = 40.3%
52 features


## Baseline de persistencia

In [3]:
results = {"persistence": T.report(real, persistence_score(real).to_numpy())}
T.summary(results)

,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
persistence,0.567032,0.532-0.577,0.47333,0.678226,0.589602,0.495204,0.395983,0.5


## Preentrenamiento en MPUD

Los pesos se guardan en `data/models/lstm_mpud.pt`; si el archivo ya existe, se cargan y se salta este paso.

In [4]:
weights = Path("data/models/lstm_mpud.pt")
if weights.exists():
    from sklearn.preprocessing import StandardScaler
    state = torch.load(weights, weights_only=False)
    pre = L.LSTMModel(L.Net(), state["scaler"])
    pre.net.load_state_dict(state["net"])
    print("loaded", weights)
else:
    pre = L.pretrain(log, mpud)
    weights.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"net": pre.net.state_dict(), "scaler": pre.scaler}, weights)

epoch 1: val logloss 0.6681
epoch 2: val logloss 0.6752
epoch 3: val logloss 0.6701
epoch 4: val logloss 0.6688


## Dejando una persona fuera

In [5]:
scores_real = L.run_loso(log, real, pre=None)
results["lstm_real_only"] = T.report(real, scores_real)
scores_mpud = L.run_loso(log, real, pre=pre)
results["lstm_mpud_finetuned"] = T.report(real, scores_mpud)
T.summary(results)

fold 1/18 done
fold 2/18 done
fold 3/18 done
fold 4/18 done
fold 5/18 done
fold 6/18 done
fold 7/18 done
fold 8/18 done
fold 9/18 done
fold 10/18 done
fold 11/18 done
fold 12/18 done
fold 13/18 done
fold 14/18 done
fold 15/18 done
fold 16/18 done
fold 17/18 done
fold 18/18 done
fold 1/18 done
fold 2/18 done
fold 3/18 done
fold 4/18 done
fold 5/18 done
fold 6/18 done
fold 7/18 done
fold 8/18 done
fold 9/18 done
fold 10/18 done
fold 11/18 done
fold 12/18 done
fold 13/18 done
fold 14/18 done
fold 15/18 done
fold 16/18 done
fold 17/18 done
fold 18/18 done


,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
persistence,0.567032,0.532-0.577,0.47333,0.678226,0.589602,0.495204,0.395983,0.5
lstm_real_only,0.51031,0.465-0.574,0.34495,0.742454,0.505029,0.433713,0.252758,0.505848
lstm_mpud_finetuned,0.547037,0.502-0.585,0.341379,0.691271,0.587537,0.52175,0.241854,0.528377


## Partición temporal (secundaria)

In [6]:
temporal = {}
test, s = L.run_temporal(log, real, pre=None)
temporal["lstm_real_only"] = T.report(test, s)
test, s = L.run_temporal(log, real, pre=pre)
temporal["lstm_mpud_finetuned"] = T.report(test, s)
temporal["persistence"] = T.report(test, persistence_score(test).to_numpy())
T.summary(temporal)

,AUC median,CI95,AUC min,AUC max,AUC pooled,PR-AUC,Brier,Onset AUC
lstm_real_only,0.586267,0.543-0.647,0.44127,0.77037,0.640697,0.56534,0.226472,0.565877
lstm_mpud_finetuned,0.516995,0.474-0.564,0.39943,0.667094,0.55267,0.434419,0.241837,0.507221
persistence,0.551023,0.500-0.583,0.451587,0.634425,0.571412,0.450331,0.401779,0.5


## Decisión E13 y E8

Compara con los resultados de `e8_lr.ipynb` y `e8_xgb.ipynb` si ya están guardados.

In [7]:
auc = {k: r["auc_median"] for k, r in results.items()}
gain = auc["lstm_mpud_finetuned"] - auc["lstm_real_only"]
best_arm = "lstm_mpud_finetuned" if gain >= 0.01 else "lstm_real_only"
print(f"E13: MPUD adds {gain:+.3f} AUC -> keep MPUD: {gain >= 0.01}; arm used: {best_arm}")
tab = {}
for name in ("lr", "xgb"):
    f = Path(f"data/reports/e8_{name}.json")
    if f.exists():
        r = json.loads(f.read_text())
        tab[name] = r["loso"][r["best_arm"]]["auc_median"]
if tab:
    best_tab = max(tab, key=tab.get)
    print(f"best tabular: {best_tab} {tab[best_tab]:.3f}; LSTM {auc[best_arm]:.3f} "
          f"-> adopt LSTM: {auc[best_arm] - tab[best_tab] >= 0.01}")

E13: MPUD adds +0.037 AUC -> keep MPUD: True; arm used: lstm_mpud_finetuned
best tabular: xgb 0.606; LSTM 0.547 -> adopt LSTM: False


## Guardar

In [8]:
out = {"loso": results, "temporal": temporal, "e13_gain": gain, "best_arm": best_arm}
Path("data/reports/e8_lstm.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e8_lstm.json")

-> data/reports/e8_lstm.json
